In [251]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_groq import ChatGroq
from langchain_core.documents import Document
from dotenv import load_dotenv

In [252]:
load_dotenv()

True

In [253]:
documents = [
    Document(page_content="The geopolitical relationship between India and China is defined by intense strategic competition and historic border disputes along the Line of Actual Control (LAC)."),
    Document(page_content="Bilateral trade between India and China has continued to grow rapidly despite diplomatic tensions, making China one of India's largest trading partners."),
    Document(page_content="Artificial Intelligence models rely heavily on high-speed hardware and advanced algorithms to process massive volumes of multi-lingual text efficiently."),
    Document(page_content="Retrieval-Augmented Generation (RAG) enhances Large Language Models by injecting relevant external source documents directly into the prompt context.")
]


In [254]:
load_dotenv()

embedding = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore = Chroma.from_documents(
    documents=documents,
    embedding= embedding,
    collection_name="my_collection"
)

In [255]:
retriever = vectorstore.as_retriever(search_kwargs={"k":2})

In [256]:
query = "What is RAG in AI"
results = retriever.invoke(query)

In [257]:
print(results)

[Document(metadata={}, page_content='Retrieval-Augmented Generation (RAG) enhances Large Language Models by injecting relevant external source documents directly into the prompt context.'), Document(metadata={}, page_content='Retrieval-Augmented Generation (RAG) enhances Large Language Models by injecting relevant external source documents directly into the prompt context.')]


In [258]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(f"Content:\n{doc.page_content}...")


--- Result 1 ---
Content:
Retrieval-Augmented Generation (RAG) enhances Large Language Models by injecting relevant external source documents directly into the prompt context....

--- Result 2 ---
Content:
Retrieval-Augmented Generation (RAG) enhances Large Language Models by injecting relevant external source documents directly into the prompt context....


MMR - Maximum Marginal Relavance
___


In [259]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [260]:
embedding_1 = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore_1 = Chroma.from_documents(
    documents=docs,
    embedding= embedding_1,
    collection_name="my_collection_1"
)

In [261]:
retriever_1 = vectorstore_1.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k":3, "lambda_mult": 0.5} #k=TOP RESULT AND LAMBDA MULT = RELEVENCE DIVERSITY BALANCE , if it 1, behave as similarity search
)

In [262]:
query_1 = "What is LanChain?"
results_1 = retriever_1.invoke(query_1)

In [263]:
results_1

[Document(metadata={}, page_content='LangChain is used to build LLM based applications.'),
 Document(metadata={}, page_content='LangChain supports Chroma, FAISS, Pinecone, and more.'),
 Document(metadata={}, page_content='MMR helps you get diverse results when doing similarity search.')]

In [264]:
for i, doc in enumerate(results_1):
    print(f"\n--- Result {i+1} ---")
    print(f"Content:\n{doc.page_content}...")


--- Result 1 ---
Content:
LangChain is used to build LLM based applications....

--- Result 2 ---
Content:
LangChain supports Chroma, FAISS, Pinecone, and more....

--- Result 3 ---
Content:
MMR helps you get diverse results when doing similarity search....


MQR - Multi Query Retriver
______

Use when user querry is not clear.....Ambigous user Querry ---> LLM --> Genarate Diverse Related Multiple Querry ---> send those querry to retrivals --- get marge that multiple querry you fetch --- remove duplicatte ---and return the tops

In [265]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [ ]:
embedding_2 = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore_2 = Chroma.from_documents(
    documents=all_docs,
    embedding= embedding_2,
    collection_name="my_collection_2"
)

In [267]:
from langchain_classic.retrievers import MultiQueryRetriever

In [274]:
similarity_retriever = vectorstore_2.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [275]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever = vectorstore_2.as_retriever(search_kwargs={"k": 3}),
    llm = ChatGroq(model="allam-2-7b")
)

In [276]:
query_2 = "How to improve energy levels and maintain balance?"

In [277]:
similarity_results = similarity_retriever.invoke(query_2)
multiquery_results= multiquery_retriever.invoke(query_2)

In [278]:
similarity_results

[Document(metadata={'source': 'H5'}, page_content='Drinking sufficient water throughout the day helps maintain metabolism and energy.'),
 Document(metadata={'source': 'H5'}, page_content='Drinking sufficient water throughout the day helps maintain metabolism and energy.'),
 Document(metadata={'source': 'H5'}, page_content='Drinking sufficient water throughout the day helps maintain metabolism and energy.')]

In [279]:
multiquery_results

[Document(metadata={'source': 'I2'}, page_content='Python balances readability with power, making it a popular system design language.'),
 Document(metadata={'source': 'I3'}, page_content='Photosynthesis enables plants to produce energy by converting sunlight.'),
 Document(metadata={'source': 'H5'}, page_content='Drinking sufficient water throughout the day helps maintain metabolism and energy.')]

CCR - Contextual Compression Retriever
____

In [280]:
#takes querry send a normal retriver --->>fetch n numbers of doc---> send them to LLM annd also send the user queery to llm ...and write a prompt autometically to trim and irrelivent things

In [283]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [284]:
# Recreate the document objects from the previous data
docs_3 = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [ ]:
embedding_3 = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2"
)
vectorstore_3 = Chroma.from_documents(
    documents=docs_3,
    embedding= embedding_3,
    collection_name="my_collection_3"
)

In [287]:
base_retriver = vectorstore_3.as_retriever(search_kwargs={"k":3})

In [286]:
llm = ChatGroq(model='allam-2-7b')
compressor = LLMChainExtractor.from_llm(llm)

In [288]:
compression_retriver = ContextualCompressionRetriever(
    base_retriever=base_retriver,
    base_compressor=compressor
)

In [291]:
query_3 = "What is photosynthesis?"
compressed_result = compression_retriver.invoke(query_3)

In [292]:
compressed_result

[Document(metadata={'source': 'Doc1'}, page_content='Photosynthesis is the process by which green plants convert sunlight into energy. \n\nTo answer the question "What is photosynthesis?" based on the provided context, we focus on the part that explains the definition of photosynthesis: "Photosynthesis is the process by which green plants convert sunlight into energy."'),
 Document(metadata={'source': 'Doc2'}, page_content='Photosynthesis is a process in plants where chlorophyll captures sunlight. \n\nTo arrive at this answer, I carefully reviewed the provided context. The question asked about the definition or explanation of photosynthesis, a topic that is briefly touched upon towards the start of the context text. I identified the relevant part as "The chlorophyll in plant cells captures sunlight during photosynthesis." This part directly answers the question about what photosynthesis is without adding or removing any information from the original context.'),
 Document(metadata={'sou

In [293]:
for i, doc in enumerate(compressed_result):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy. 

To answer the question "What is photosynthesis?" based on the provided context, we focus on the part that explains the definition of photosynthesis: "Photosynthesis is the process by which green plants convert sunlight into energy."

--- Result 2 ---
Photosynthesis is a process in plants where chlorophyll captures sunlight. 

To arrive at this answer, I carefully reviewed the provided context. The question asked about the definition or explanation of photosynthesis, a topic that is briefly touched upon towards the start of the context text. I identified the relevant part as "The chlorophyll in plant cells captures sunlight during photosynthesis." This part directly answers the question about what photosynthesis is without adding or removing any information from the original context.

--- Result 3 ---
Photosynthesis does not occur in animal cells. 

To arrive at this answer, I carefully